In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

## Getting the Raw data in a DataFrame

In [10]:
df1 = pd.read_json("MyData\StreamingHistory0.json")
df2 = pd.read_json("MyData\StreamingHistory1.json")

In [11]:
df = pd.concat([df1, df2])

In [12]:
df.reset_index(drop=True, inplace=True)

In [12]:
df.to_csv("MyData\StreamingHistory.csv", index=False)

## Some Preprocessing

### Minutes Played

In [13]:
df["Minutes Played"] = np.round(df["msPlayed"]/60000, 2)

In [25]:
df.head()

,endTime,artistName,trackName,msPlayed,Day,Week,Month,Minutes Played
0,2021-01-06 13:11:00,Taylor Swift,exile (feat. Bon Iver),42512,6,1,1,0.71
1,2021-01-06 13:16:00,Taylor Swift,willow - dancing witch version (Elvira remix),251,6,1,1,0.00
2,2021-01-06 13:17:00,Taylor Swift,willow,19600,6,1,1,0.33
3,2021-01-06 13:22:00,Taylor Swift,happiness,1004,6,1,1,0.02
4,2021-01-06 13:22:00,Taylor Swift,tolerate it,245440,6,1,1,4.09


### Adding Day, Week and Month

In [14]:
df["endTime"] = pd.to_datetime(df["endTime"])

In [15]:
df['Day'] = df['endTime'].dt.day
df["Week"] = df['endTime'].dt.week
df["Month"] = df["endTime"].dt.month

<ipython-input-15-d4f884928690>:2: FutureWarning: Series.dt.weekofyear and Series.dt.week have been deprecated.  Please use Series.dt.isocalendar().week instead.
  df["Week"] = df['endTime'].dt.week


In [26]:
df.head()

,endTime,artistName,trackName,msPlayed,Day,Week,Month,Minutes Played
0,2021-01-06 13:11:00,Taylor Swift,exile (feat. Bon Iver),42512,6,1,1,0.71
1,2021-01-06 13:16:00,Taylor Swift,willow - dancing witch version (Elvira remix),251,6,1,1,0.00
2,2021-01-06 13:17:00,Taylor Swift,willow,19600,6,1,1,0.33
3,2021-01-06 13:22:00,Taylor Swift,happiness,1004,6,1,1,0.02
4,2021-01-06 13:22:00,Taylor Swift,tolerate it,245440,6,1,1,4.09


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19556 entries, 0 to 19555
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   endTime         19556 non-null  datetime64[ns]
 1   artistName      19556 non-null  object        
 2   trackName       19556 non-null  object        
 3   msPlayed        19556 non-null  int64         
 4   Day             19556 non-null  int64         
 5   Week            19556 non-null  int64         
 6   Month           19556 non-null  int64         
 7   Minutes Played  19556 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(4), object(2)
memory usage: 1.2+ MB


### Dropping Unnecessary Columns and saving the data

In [16]:
df.drop("msPlayed", axis=1, inplace=True)

In [17]:
extras_df = df[(df["Minutes Played"]>0.5)].reset_index(drop=True)

In [39]:
extras_df.to_csv("MyData\StreamingHistoryExtras.csv", index=False)

## Grouping Data

In [32]:
extras_df = pd.read_csv("MyData\StreamingHistoryExtras.csv")

### By Artist

In [33]:
all_artists = extras_df["artistName"].unique()
len(all_artists)

125

We'll drop all artists that have have less than  hour of runtime.

In [34]:
df_grouped_by_artists = extras_df.groupby("artistName").sum()["Minutes Played"]
len(df_grouped_by_artists)

125

In [35]:
top_artists = df_grouped_by_artists[df_grouped_by_artists>60].sort_values(ascending=False).index.tolist()
len(top_artists)

31

In [36]:
top_artists

['Taylor Swift',
 'Céline Dion',
 'Ed Sheeran',
 'Shawn Mendes',
 'Alan Walker',
 'Lata Mangeshkar',
 'Camila Cabello',
 'Adele',
 'Lady Gaga',
 'Maren Morris',
 'Christina Perri',
 'Dua Lipa',
 'Miranda Lambert',
 'Justin Bieber',
 'Lea Salonga',
 'Britney Spears',
 'Selena Gomez',
 'Mandy Moore',
 'Lorne Balfe',
 'Harry Styles',
 'Idina Menzel',
 'Billie Eilish',
 'Mohammed Rafi',
 'Tape Machines',
 'Mukesh',
 'Susan Egan',
 'Asha Bhosle',
 'Kishore Kumar',
 'Peabo Bryson',
 'Ludwig van Beethoven',
 'Queen']

In [37]:
artists_dict = {}
for artist in top_artists:
    artists_dict[artist] = extras_df[extras_df["artistName"]==artist]

In [38]:
for artist in artists_dict.keys():
    artist_df = artists_dict[artist].reset_index(drop=True)
    artist_df.to_csv(f"MyData\{artist}.csv", index=False)

### By Month

In [46]:
month_dict = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}
for i in range(1,13):
    month_df = extras_df[extras_df["Month"]==i].reset_index(drop=True)
    month_df.to_csv(f"MyData\{month_dict[i]}.csv", index=False)

In [49]:
extras_df[extras_df["Month"]==3]

,endTime,artistName,trackName,Day,Week,Month,Minutes Played
3801,2021-03-01 01:56:00,Alan Walker,Darkside,1,9,3,1.36
3802,2021-03-01 02:01:00,Ed Sheeran,Perfect,1,9,3,4.39
3803,2021-03-01 02:05:00,Ed Sheeran,Shape of You,1,9,3,3.90
3804,2021-03-01 02:08:00,Justin Bieber,Anyone,1,9,3,3.18
3805,2021-03-01 02:11:00,Shawn Mendes,Where Were You In The Morning?,1,9,3,3.34
...,...,...,...,...,...,...,...
5918,2021-03-31 14:16:00,Alan Walker,Darkside,31,13,3,3.53
5919,2021-03-31 14:20:00,Taylor Swift,Fearless,31,13,3,4.03
5920,2021-03-31 14:24:00,Céline Dion,I'm Alive,31,13,3,3.50
5921,2021-03-31 14:29:00,Taylor Swift,evermore (feat. Bon Iver),31,13,3,5.07
